# **MobileNetV2 Quantization and Layer Sensitivity**

# ECE 662 — Team 9

**Owner**: Lilly Press

**Goal**: Compare 8-bit and 4-bit weight quantization, then identify which layers lose the most accuracy when reduced to 4 bits.  

**Current stage**: Uniform weight quantization. Activations remain FP32 during initial debugging; final experiments will use fixed 8-bit activations.

# 1. Setup and Imports

In [1]:
import copy
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

# Set seeds for reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Use a GPU if available; otherwise use the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"Device: {device}")

if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.11.0+cpu
Torchvision version: 0.26.0+cpu
Device: cpu


# 2. Load the Baseline Model

In [2]:
# Load MobileNetV2 pretrained on CIFAR-10
MODEL_REPO = "chenyaofo/pytorch-cifar-models"
MODEL_NAME = "cifar10_mobilenetv2_x1_0"

baseline_model = torch.hub.load(
    MODEL_REPO,
    MODEL_NAME,
    pretrained=True,
    trust_repo=True
)

baseline_model = baseline_model.to(device)
baseline_model.eval()

# Keep baseline_model unchanged.
# Later, we will make copies for quantization.

# Check that the model accepts a CIFAR-10-sized image
with torch.inference_mode():
    example_input = torch.zeros(1, 3, 32, 32, device=device)
    example_output = baseline_model(example_input)

assert example_output.shape == (1, 10)

num_parameters = sum(
    parameter.numel() for parameter in baseline_model.parameters()
)

print(f"Model: {MODEL_NAME}")
print(f"Device: {device}")
print(f"Parameters: {num_parameters:,}")
print(f"Output shape: {tuple(example_output.shape)}")
print("Baseline model loaded successfully.")

Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/zipball/master" to /root/.cache/torch/hub/master.zip
Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/releases/download/mobilenetv2/cifar10_mobilenetv2_x1_0-fe6a5b48.pt" to /root/.cache/torch/hub/checkpoints/cifar10_mobilenetv2_x1_0-fe6a5b48.pt


100%|██████████| 8.77M/8.77M [00:00<00:00, 128MB/s]


Model: cifar10_mobilenetv2_x1_0
Device: cpu
Parameters: 2,236,682
Output shape: (1, 10)
Baseline model loaded successfully.


# 3. Load CIFAR-10

In [3]:
# Convert images to tensors and normalize each color channel
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2023, 0.1994, 0.2010)
    )
])

# Download the CIFAR-10 test set
test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# Full test set for reporting final accuracy
test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

# Small test subset for quick debugging only
generator = torch.Generator().manual_seed(SEED)
debug_indices = torch.randperm(
    len(test_dataset), generator=generator
)[:1000].tolist()

debug_loader = DataLoader(
    Subset(test_dataset, debug_indices),
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

print(f"Full test set: {len(test_dataset):,} images")
print(f"Debug subset: {len(debug_indices):,} images")
print(f"Classes: {test_dataset.classes}")

100%|██████████| 170M/170M [47:34<00:00, 59.7kB/s]


Full test set: 10,000 images
Debug subset: 1,000 images
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


# 4. Check FP32 Accuracy

In [4]:
# Reuse this function for every model configuration
def evaluate(model, loader):
    model.eval()
    model_device = next(model.parameters()).device

    correct = 0
    total = 0

    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(model_device)
            labels = labels.to(model_device)

            outputs = model(images)
            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    if total == 0:
        raise ValueError("The data loader contains no images.")

    return 100.0 * correct / total


# Start with the smaller subset to check everything works
fp32_debug_accuracy = evaluate(baseline_model, debug_loader)
print(f"FP32 debug accuracy: {fp32_debug_accuracy:.2f}%")

FP32 debug accuracy: 94.20%


In [5]:
fp32_test_accuracy = evaluate(baseline_model, test_loader)
print(f"FP32 full-test accuracy: {fp32_test_accuracy:.2f}%")

FP32 full-test accuracy: 94.05%


# 5. Define Weight Quantization

In [6]:
def quantize_weights(weights, bits):
    """Simulate symmetric quantization per output channel."""
    if bits not in (4, 8):
        raise ValueError("bits must be 4 or 8")

    # Symmetric integer range: -7 to 7 or -127 to 127
    qmax = (2 ** (bits - 1)) - 1

    # Dimension 0 represents output channels for Conv2d and Linear.
    # Find the maximum magnitude within each output channel.
    reduce_dims = tuple(range(1, weights.ndim))
    max_abs = weights.abs().amax(dim=reduce_dims, keepdim=True)

    # Use scale 1 for all-zero channels to avoid division by zero
    scale = torch.where(
        max_abs > 0,
        max_abs / qmax,
        torch.ones_like(max_abs)
    )

    integer_weights = torch.round(weights / scale)
    integer_weights = torch.clamp(integer_weights, -qmax, qmax)

    # Convert back to floating point for ordinary PyTorch inference
    return integer_weights * scale


def quantize_model_weights(model, bits):
    """Return a quantized copy, leaving the original model unchanged."""
    quantized_model = copy.deepcopy(model)
    quantized_model.eval()

    with torch.no_grad():
        for name, layer in quantized_model.named_modules():
            if isinstance(layer, (nn.Conv2d, nn.Linear)):
                layer.weight.copy_(
                    quantize_weights(layer.weight, bits)
                )

    return quantized_model


print("Weight quantization functions are ready.")

Weight quantization functions are ready.


# 6. Run Uniform 8-bit Weights

In [7]:
# Create an 8-bit weight version from the untouched baseline
model_w8 = quantize_model_weights(baseline_model, bits=8)

# Check accuracy on the same debugging subset as FP32
w8_debug_accuracy = evaluate(model_w8, debug_loader)

print("Configuration: W8 / FP32 activations")
print(f"FP32 debug accuracy: {fp32_debug_accuracy:.2f}%")
print(f"W8 debug accuracy:   {w8_debug_accuracy:.2f}%")
print(
    f"Accuracy drop: "
    f"{fp32_debug_accuracy - w8_debug_accuracy:.2f} percentage points"
)

Configuration: W8 / FP32 activations
FP32 debug accuracy: 94.20%
W8 debug accuracy:   94.30%
Accuracy drop: -0.10 percentage points


In [8]:
w8_test_accuracy = evaluate(model_w8, test_loader)

print(f"W8 full-test accuracy: {w8_test_accuracy:.2f}%")
print(
    f"Accuracy drop from FP32: "
    f"{fp32_test_accuracy - w8_test_accuracy:.2f} percentage points"
)

W8 full-test accuracy: 94.02%
Accuracy drop from FP32: 0.03 percentage points


# 7. Run Uniform 4-bit Weights

In [9]:
# Create a fresh 4-bit version from the original FP32 model
model_w4 = quantize_model_weights(baseline_model, bits=4)

w4_debug_accuracy = evaluate(model_w4, debug_loader)

print("Configuration: W4 / FP32 activations")
print(f"FP32 debug accuracy: {fp32_debug_accuracy:.2f}%")
print(f"W4 debug accuracy:   {w4_debug_accuracy:.2f}%")
print(
    f"Accuracy drop: "
    f"{fp32_debug_accuracy - w4_debug_accuracy:.2f} percentage points"
)

Configuration: W4 / FP32 activations
FP32 debug accuracy: 94.20%
W4 debug accuracy:   93.10%
Accuracy drop: 1.10 percentage points


In [12]:
w4_test_accuracy = evaluate(model_w4, test_loader)

print(f"W4 full-test accuracy: {w4_test_accuracy:.2f}%")
print(
    f"Accuracy drop from FP32: "
    f"{fp32_test_accuracy - w4_test_accuracy:.2f} percentage points"
)

W4 full-test accuracy: 92.84%
Accuracy drop from FP32: 1.21 percentage points


# 8. Calculate Estimated Weight Storage

In [13]:
def estimate_weight_storage(model, bits):
    """Packed Conv2d/Linear weight payload only, in decimal MB."""
    weight_count = sum(
        layer.weight.numel()
        for layer in model.modules()
        if isinstance(layer, (nn.Conv2d, nn.Linear))
    )

    return weight_count * bits / 8 / 1_000_000


storage_results = pd.DataFrame({
    "Weight precision": ["FP32", "8-bit", "4-bit"],
    "Estimated weight payload (MB)": [
        estimate_weight_storage(baseline_model, 32),
        estimate_weight_storage(baseline_model, 8),
        estimate_weight_storage(baseline_model, 4)
    ]
})

print(storage_results.to_string(index=False))

Weight precision  Estimated weight payload (MB)
            FP32                        8.81024
           8-bit                        2.20256
           4-bit                        1.10128


# 9. Compare Baseline Results

In [14]:
baseline_results = pd.DataFrame({
    "Configuration": [
        "FP32",
        "W8 / FP32 activations",
        "W4 / FP32 activations"
    ],
    "Test accuracy (%)": [
        fp32_test_accuracy,
        w8_test_accuracy,
        w4_test_accuracy
    ],
    "Drop from FP32 (percentage points)": [
        0.0,
        fp32_test_accuracy - w8_test_accuracy,
        fp32_test_accuracy - w4_test_accuracy
    ],
    "Estimated weight payload (MB)": [
        estimate_weight_storage(baseline_model, 32),
        estimate_weight_storage(baseline_model, 8),
        estimate_weight_storage(baseline_model, 4)
    ]
})

display(baseline_results.round(3))

,Configuration,Test accuracy (%),Drop from FP32 (percentage points),Estimated weight payload (MB)
0,FP32,94.05,0.00,8.810
1,W8 / FP32 activations,94.02,0.03,2.203
2,W4 / FP32 activations,92.84,1.21,1.101


# 10. Add per-layer precision

In [15]:
# Names of all layers eligible for weight quantization
eligible_layers = [
    name
    for name, layer in baseline_model.named_modules()
    if isinstance(layer, (nn.Conv2d, nn.Linear))
]


def quantize_model_by_layer(model, bit_config):
    """Quantize a fresh model copy using a layer-name-to-bits dictionary."""
    model_layers = {
        name: layer
        for name, layer in model.named_modules()
        if isinstance(layer, (nn.Conv2d, nn.Linear))
    }

    # Require a precision assignment for every eligible layer
    if set(bit_config) != set(model_layers):
        raise ValueError(
            "bit_config must contain exactly the eligible layer names."
        )

    if any(bits not in (4, 8) for bits in bit_config.values()):
        raise ValueError("Each layer must use 4 or 8 bits.")

    quantized_model = copy.deepcopy(model)
    quantized_model.eval()

    with torch.no_grad():
        for name, layer in quantized_model.named_modules():
            if name in bit_config:
                layer.weight.copy_(
                    quantize_weights(layer.weight, bit_config[name])
                )

    return quantized_model


# Start with every eligible layer at 8 bits
all_w8_config = {name: 8 for name in eligible_layers}

# Example: change only the first eligible layer to 4 bits
example_config = all_w8_config.copy()
example_config[eligible_layers[0]] = 4

print(f"Eligible layers: {len(eligible_layers)}")
print(f"Example 4-bit layer: {eligible_layers[0]}")
print("Per-layer quantization function is ready.")

Eligible layers: 53
Example 4-bit layer: features.0.0
Per-layer quantization function is ready.


In [16]:
# Build an all-8-bit model using the per-layer configuration
model_w8_by_layer = quantize_model_by_layer(
    baseline_model, all_w8_config
)

# Its weights and other state should match the existing W8 model
for name, value in model_w8.state_dict().items():
    assert torch.equal(
        value, model_w8_by_layer.state_dict()[name]
    ), f"Mismatch in: {name}"

print("Passed: per-layer all-W8 matches the uniform W8 model.")

# Check that the example mixed configuration runs
example_mixed_model = quantize_model_by_layer(
    baseline_model, example_config
)

example_debug_accuracy = evaluate(
    example_mixed_model, debug_loader
)

print(f"4-bit layer: {eligible_layers[0]}")
print(f"All other eligible layers: 8 bits")
print(f"Debug accuracy: {example_debug_accuracy:.2f}%")

Passed: per-layer all-W8 matches the uniform W8 model.
4-bit layer: features.0.0
All other eligible layers: 8 bits
Debug accuracy: 93.70%


# 11. Fold Batch Normalization

In [17]:
from torch.nn.utils import fuse_conv_bn_eval

def fold_batch_norm(model):
    folded_model = copy.deepcopy(model).eval()
    folded_pairs = []

    def fold_sequential_pairs(module, prefix=""):
        # Recurse into nested modules
        for name, child in list(module.named_children()):
            child_prefix = f"{prefix}.{name}" if prefix else name
            fold_sequential_pairs(child, child_prefix)

        # Only combine adjacent layers in Sequential containers
        if isinstance(module, nn.Sequential):
            children = list(module.named_children())

            for i in range(len(children) - 1):
                conv_name, conv = children[i]
                bn_name, bn = children[i + 1]

                if isinstance(conv, nn.Conv2d) and isinstance(
                    bn, nn.BatchNorm2d
                ):
                    module._modules[conv_name] = fuse_conv_bn_eval(
                        conv, bn
                    )
                    module._modules[bn_name] = nn.Identity()
                    folded_pairs.append(
                        f"{prefix}.{conv_name}" if prefix else conv_name
                    )

    fold_sequential_pairs(folded_model)

    remaining_bn = sum(
        isinstance(layer, nn.BatchNorm2d)
        for layer in folded_model.modules()
    )
    if remaining_bn:
        raise ValueError(
            f"{remaining_bn} batch-normalization layers remain; "
            "check the model structure."
        )

    return folded_model, folded_pairs


# Preserve the original baseline
folded_baseline, folded_pairs = fold_batch_norm(baseline_model)

# Compare outputs on one real image batch
images, _ = next(iter(debug_loader))
images = images.to(next(baseline_model.parameters()).device)

with torch.inference_mode():
    original_outputs = baseline_model(images)
    folded_outputs = folded_baseline(images)

max_difference = (
    original_outputs - folded_outputs
).abs().max().item()

torch.testing.assert_close(
    folded_outputs,
    original_outputs,
    rtol=1e-3,
    atol=1e-4
)

print(f"Folded convolution/BN pairs: {len(folded_pairs)}")
print(f"Maximum output difference: {max_difference:.8f}")
print("Passed: folding preserves FP32 outputs within tolerance.")

Folded convolution/BN pairs: 52
Maximum output difference: 0.00000858
Passed: folding preserves FP32 outputs within tolerance.
